# 02 - Debug Streaming Lag

Diagnose latency between Kafka ingestion and Delta table landing.
Useful when consumer lag alerts fire or data freshness checks fail.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("debug-lag")
    .config("spark.jars.packages", "io.delta:delta-spark_2.12:3.2.0")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.hadoop.fs.s3a.endpoint", "http://localhost:9000")
    .config("spark.hadoop.fs.s3a.access.key", "minioadmin")
    .config("spark.hadoop.fs.s3a.secret.key", "minioadmin")
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .getOrCreate()
)

In [ ]:
# Silver trades: check last record timestamp vs now
silver_trades = spark.read.format("delta").load("s3a://cryptostream-dev-silver/trades/")

freshness = (
    silver_trades
    .groupBy("symbol")
    .agg(
        F.max("ingested_at").alias("latest_ingested"),
        F.max((F.col("event_time") / 1000).cast("timestamp")).alias("latest_event"),
        F.count("*").alias("total_rows"),
    )
    .withColumn("lag_seconds",
        F.unix_timestamp(F.current_timestamp()) - F.unix_timestamp("latest_ingested")
    )
)

freshness.show(truncate=False)

In [ ]:
# Latency distribution: event_time → ingested_at
latency = (
    silver_trades
    .withColumn("event_ts", (F.col("event_time") / 1000).cast("timestamp"))
    .withColumn("ingested_ts", F.to_timestamp("ingested_at"))
    .withColumn("processing_latency_s",
        F.unix_timestamp("ingested_ts") - F.unix_timestamp("event_ts")
    )
    .filter(F.col("processing_latency_s").isNotNull())
)

latency.select("processing_latency_s").describe().show()

# Percentiles
percentiles = latency.stat.approxQuantile("processing_latency_s", [0.5, 0.9, 0.95, 0.99], 0.01)
print(f"P50: {percentiles[0]}s, P90: {percentiles[1]}s, P95: {percentiles[2]}s, P99: {percentiles[3]}s")

In [ ]:
# Check for partition skew by symbol
(
    silver_trades
    .groupBy("symbol")
    .agg(
        F.count("*").alias("row_count"),
        F.min("ingested_at").alias("earliest"),
        F.max("ingested_at").alias("latest"),
    )
    .withColumn("pct_of_total",
        F.round(F.col("row_count") / F.sum("row_count").over(F.Window.partitionBy()), 3)
    )
    .show(truncate=False)
)

In [ ]:
# Check checkpoint lag: compare Kafka offset in Bronze vs latest
bronze = spark.read.format("delta").load("s3a://cryptostream-dev-bronze/raw_events/")

(
    bronze
    .groupBy("kafka_topic", "kafka_partition")
    .agg(
        F.max("kafka_offset").alias("max_offset"),
        F.max("kafka_timestamp").alias("latest_kafka_ts"),
    )
    .orderBy("kafka_topic", "kafka_partition")
    .show(truncate=False)
)

In [ ]:
spark.stop()